In [ ]:
import sys, pathlib, os

# Ensure repository root on sys.path as required
try:
    sys.path.append(str(pathlib.Path(__file__).resolve().parents[1]))
    repo_root = pathlib.Path(__file__).resolve().parents[1]
except NameError:
    nb_guess = pathlib.Path(os.getcwd()) / "notebooks" / "03_forecast.ipynb"
    repo_root = nb_guess.resolve().parents[1] if nb_guess.exists() else pathlib.Path(os.getcwd()).resolve()
    sys.path.append(str(repo_root))

print(f"Repository root: {repo_root}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.plotting import set_matplotlib_style
from utils.run import RunContext
import utils.config as config

from stages.forecast import run_forecast

# Set plotting style for downstream figures
set_matplotlib_style()

In [ ]:
import json, yaml
from pathlib import Path

cfg_paths = [repo_root / "configs" / "default.yaml", repo_root / "configs" / "forecast.yaml"]
schema_path = repo_root / "configs" / "schema.json"

# Load and merge configs using utils.config if available
cfg = None
try:
    if hasattr(config, "load_config"):
        cfg = config.load_config(paths=[str(p) for p in cfg_paths])
    elif hasattr(config, "load_and_validate"):
        cfg = config.load_and_validate(paths=[str(p) for p in cfg_paths], schema_path=str(schema_path))
except Exception as e:
    print(f"utils.config helper failed: {e}")

if cfg is None:
    # Fallback: YAML read and shallow merge (later keys override earlier)
    merged = {}
    for p in cfg_paths:
        with open(p, "r", encoding="utf-8") as f:
            d = yaml.safe_load(f) or {}
        for k, v in d.items():
            if isinstance(v, dict) and k in merged and isinstance(merged[k], dict):
                merged[k].update(v)
            else:
                merged[k] = v
    cfg = merged

# Validate against schema using utils.config or jsonschema
try:
    if hasattr(config, "validate_config"):
        config.validate_config(cfg, schema_path=str(schema_path))
    else:
        import jsonschema
        with open(schema_path, "r", encoding="utf-8") as f:
            schema = json.load(f)
        jsonschema.validate(instance=cfg, schema=schema)
except Exception as e:
    raise

print("Configuration loaded and validated for forecast stage.")

In [ ]:
from utils.plotting import place_legend_below  # ensure available for downstream plots
import os, random

stage_name = "forecast"

# Start run and stage contexts
run = RunContext.start(cfg)
ctx = run.stage(stage_name)

# Set deterministic seeds
seed = int(cfg.get("seed", 12345))
try:
    from utils.seeds import set_global_seeds
    set_global_seeds(seed)
except Exception:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)

# Optional: structured log at start
try:
    if hasattr(ctx, "log_json"):
        ctx.log_json(level="INFO", stage=stage_name, site_id=None, lineage=None, message="Starting forecast stage", context={"seed": seed})
except Exception:
    pass

In [ ]:
# Run the Forecast stage (PF + MAP + t+1 predictive)
run_forecast(cfg, ctx)

# Close the stage context with provenance
inputs = [str(p) for p in cfg_paths]
notes = "Forecast stage orchestrated via notebooks/03_forecast.ipynb"
try:
    ctx.close(inputs=inputs, notes=notes)
except Exception:
    pass

print("Forecast stage completed.")

In [ ]:
import os, time, math
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import betabinom, norm, kstest, chisquare, skew, kurtosis

# -------- Hard paths --------
BASE   = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
IN_CSV = PRIORS / "priors_full_detail.csv"
OUTDIR = PRIORS / "metr23r4cx3fic"   # keep folder name as provided
OUTDIR.mkdir(parents=True, exist_ok=True)

# -------- Speed knobs (tweak freely) --------
SEED          = 42
SAMPLE_N      = 50_000         # total rows to evaluate (stratified)
STRATA        = 10             # lambda deciles
SMALL_N_MAX   = 5_000          # exact ZIBB if n <= this; else normal approx
LEVELS        = (0.50, 0.90, 0.95)
PIT_BINS      = 10             # fewer bins => faster
EPS           = 1e-12

np.seterr(all="ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# -------- Helpers --------
def _clip01(x, eps=1e-12): 
    return np.clip(np.asarray(x, float), eps, 1.0 - eps)

def _save(df, name): 
    p = OUTDIR / name
    df.to_csv(p, index=False, float_format="%.6g")
    print("[csv]", p)

def _bb_moments(n, mu, kappa):
    a = _clip01(mu) * np.clip(kappa, EPS, np.inf)
    b = (1.0 - _clip01(mu)) * np.clip(kappa, EPS, np.inf)
    m = n * mu
    v = n * mu * (1.0 - mu) * ((a + b + n) / (a + b + 1.0))
    return m, np.maximum(v, EPS)

def _mix_moments(n, mu, kappa, pi):
    m_bb, v_bb = _bb_moments(n, mu, kappa)
    m = (1 - pi) * m_bb
    v = (1 - pi) * v_bb + (pi * (1 - pi)) * (m_bb ** 2)
    return m, np.maximum(v, EPS)

# exact mixture CDF at integer y (only for small n)
def _mix_cdf_exact(y, n, mu, kappa, pi):
    a = _clip01(mu) * np.clip(kappa, EPS, np.inf)
    b = (1.0 - _clip01(mu)) * np.clip(kappa, EPS, np.inf)
    Fbb = betabinom.cdf(np.clip(y, -1, None), n, a, b)
    return np.clip(np.where(y < 0, 0.0, pi + (1 - pi) * Fbb), 0.0, 1.0)

# continuity-corrected normal approximation for BB + zero-inflation
def _mix_cdf_normal_cc(y, n, mu, kappa, pi):
    m_bb, v_bb = _bb_moments(n, mu, kappa)
    sd = np.sqrt(v_bb)
    zhi = (y + 0.5 - m_bb) / sd
    return np.clip(pi + (1 - pi) * norm.cdf(zhi), 0.0, 1.0)

def _mix_ppf_exact(q, n, mu, kappa, pi):
    qprime = (q - pi) / np.maximum(1 - pi, EPS)
    qprime = np.clip(qprime, 0.0, 1.0 - EPS)
    a = _clip01(mu) * np.clip(kappa, EPS, np.inf)
    b = (1.0 - _clip01(mu)) * np.clip(kappa, EPS, np.inf)
    y = betabinom.ppf(qprime, n, a, b).astype(int)
    return np.where(q <= pi, 0, y)

def _mix_ppf_normal_cc(q, n, mu, kappa, pi):
    qprime = (q - pi) / np.maximum(1 - pi, EPS)
    qprime = np.clip(qprime, 0.0, 1.0 - EPS)
    m_bb, v_bb = _bb_moments(n, mu, kappa)
    sd = np.sqrt(v_bb)
    zq = norm.ppf(qprime)
    y = np.floor(m_bb + sd * zq - 0.5).astype(int)
    y = np.where(q <= pi, 0, y)
    return np.clip(y, 0, n)

# randomized discrete PIT using chosen CDFs
def _rand_pit(y, cdf_fn):
    Fy  = cdf_fn(y)
    Fym = cdf_fn(y - 1)
    rng = np.random.default_rng(SEED)
    return np.clip(Fym + rng.random(len(y)) * np.maximum(Fy - Fym, 0.0), 0.0, 1.0)

# -------- Load & stratified sample --------
t0 = time.time()
df = pd.read_csv(IN_CSV, low_memory=False)
if "mu" in df.columns and "mu_t" not in df.columns: df.rename(columns={"mu":"mu_t"}, inplace=True)
if "kappa" in df.columns and "kappa_t" not in df.columns: df.rename(columns={"kappa":"kappa_t"}, inplace=True)
if "pi" not in df.columns: df["pi"] = 0.0

for c in ["count","coverage","mu_t","kappa_t","pi"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["count","coverage","mu_t","kappa_t","pi"]).copy()
df["count"]    = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)
df["count"]    = np.clip(df["count"], 0, df["coverage"])
df["mu_t"]     = _clip01(df["mu_t"])
df["kappa_t"]  = np.clip(df["kappa_t"], EPS, 1e12)
df["pi"]       = _clip01(df["pi"])

y  = df["count"].to_numpy(int)
n  = df["coverage"].to_numpy(int)
mu = df["mu_t"].to_numpy(float)
k  = df["kappa_t"].to_numpy(float)
pi = df["pi"].to_numpy(float)
N  = len(df)

lam = n * mu
labels = pd.qcut(lam, q=STRATA, duplicates="drop")
df["_lam_decile"] = labels.astype(str)

# proportional sample across deciles
rng = np.random.default_rng(SEED)
take_idx = []
for _, grp in df.groupby("_lam_decile", sort=False):
    gidx = grp.index.to_numpy()
    gn = len(gidx)
    want = max(1, int(round(gn / N * SAMPLE_N)))
    if want >= gn:
        take_idx.append(gidx)
    else:
        take_idx.append(rng.choice(gidx, size=want, replace=False))
take_idx = np.concatenate(take_idx)
take_idx.sort()

# sampled arrays
ys, ns, mus, kappa_s, pis = y[take_idx], n[take_idx], mu[take_idx], k[take_idx], pi[take_idx]
Ns = len(take_idx)

# choose per-row engine (exact for small n, approx for big n)
small_mask = (ns <= SMALL_N_MAX)

def cdf_mix(yv):
    out = np.empty_like(mus, dtype=float)
    if small_mask.any():
        out[small_mask] = _mix_cdf_exact(yv[small_mask], ns[small_mask], mus[small_mask], kappa_s[small_mask], pis[small_mask])
    if (~small_mask).any():
        out[~small_mask] = _mix_cdf_normal_cc(yv[~small_mask], ns[~small_mask], mus[~small_mask], kappa_s[~small_mask], pis[~small_mask])
    return out

def ppf_mix(q):
    out = np.empty_like(ns, dtype=int)
    if small_mask.any():
        out[small_mask] = _mix_ppf_exact(q, ns[small_mask], mus[small_mask], kappa_s[small_mask], pis[small_mask])
    if (~small_mask).any():
        out[~small_mask] = _mix_ppf_normal_cc(q, ns[~small_mask], mus[~small_mask], kappa_s[~small_mask], pis[~small_mask])
    return np.clip(out, 0, ns)

# moments and z on sample rows (mixture)
m_mix, v_mix = _mix_moments(ns, mus, kappa_s, pis)
sd_mix = np.sqrt(v_mix)
resid = ys - m_mix
z = resid / sd_mix

# -------- Coverage (sample) --------
# -------- Coverage (sample) --------
cov_rows = []
for lvl in LEVELS:
    qlo, qhi = (1 - lvl)/2, (1 + lvl)/2
    lo = ppf_mix(qlo)
    hi = ppf_mix(qhi)
    inside = (ys >= lo) & (ys <= hi)
    cov_rows.append({
        "nominal": float(lvl),
        "empirical": float(inside.mean()),
        "bias": float(inside.mean() - lvl),
        "lower_miss": float((ys < lo).mean()),
        "upper_miss": float((ys > hi).mean()),
        "mean_width": float((hi - lo).mean()),
        "median_width": float(np.median(hi - lo)),
        "sample_n": int(Ns),
        "population_N": int(N),
        "sampling": f"stratified_lambda_deciles={STRATA}, SMALL_N_MAX={SMALL_N_MAX}, seed={SEED}"
    })
save_cov = pd.DataFrame(cov_rows)  # sanity handle in case you want to inspect quickly in memory
_save(save_cov, "priors_predictive_coverage_quick.csv")

# -------- PIT (sample) --------
U = _rand_pit(ys, lambda yy: cdf_mix(yy))
hist, edges = np.histogram(U, bins=PIT_BINS, range=(0,1))
expected = Ns / PIT_BINS
_save(pd.DataFrame({
    "bin_left": edges[:-1], "bin_right": edges[1:],
    "count": hist, "density": hist / hist.sum() if hist.sum() else 0,
    "expected_count_uniform": expected
}), "priors_pit_uniformity_bins_quick.csv")

ks_res  = kstest(U, "uniform")
chi_res = chisquare(hist, f_exp=np.full_like(hist, expected, dtype=float))
_save(pd.DataFrame([
    {"test":"KS (Uniform)", "statistic": float(getattr(ks_res,"statistic", ks_res[0])), "pvalue": float(getattr(ks_res,"pvalue", ks_res[1])), "N": int(Ns), "bins": PIT_BINS},
    {"test":"Chi-square (Uniform bins)", "statistic": float(getattr(chi_res,"statistic", chi_res[0])), "pvalue": float(getattr(chi_res,"pvalue", chi_res[1])), "N": int(Ns), "bins": PIT_BINS}
]), "priors_pit_tests_quick.csv")

# -------- Fit diagnostics (sample) --------
vr = (resid**2) / v_mix
fit = pd.DataFrame([{
    "N_sample": int(Ns),
    "N_population": int(N),
    "rmsz": float(np.sqrt(np.mean(z**2))),
    "z_mean": float(np.mean(z)),
    "z_std": float(np.std(z)),
    "z_skew": float(skew(z, nan_policy="omit")),
    "z_kurtosis_fisher": float(kurtosis(z, fisher=True, nan_policy="omit")),
    "variance_ratio_mean": float(np.mean(vr[np.isfinite(vr)])),
    "variance_ratio_median": float(np.median(vr[np.isfinite(vr)])),
    "variance_ratio_q05": float(np.quantile(vr[np.isfinite(vr)], 0.05)),
    "variance_ratio_q95": float(np.quantile(vr[np.isfinite(vr)], 0.95)),
}])
_save(fit, "priors_fit_diagnostics_quick.csv")

# -------- Sampling report --------
rep = pd.DataFrame({
    "lambda_decile": df.loc[take_idx, "_lam_decile"].astype(str).to_numpy(),
    "n": ns, "mu": mus, "kappa": kappa_s, "pi": pis
}).groupby("lambda_decile", dropna=False).agg(
    rows=("n","size"),
    n_median=("n","median"),
    mu_median=("mu","median"),
    kappa_median=("kappa","median"),
    pi_median=("pi","median")
).reset_index()

meta = pd.DataFrame([{
    "population_N": int(N),
    "sample_N": int(Ns),
    "strata": STRATA,
    "seed": SEED,
    "small_n_threshold": SMALL_N_MAX
}])

_save(rep,  "priors_sampling_profile_quick.csv")
_save(meta, "priors_sampling_meta_quick.csv")

print(f"[done QUICK] popN={N:,} sampleN={Ns:,} | wrote *_quick.csv to {OUTDIR} | {time.time()-t0:.1f}s")


SyntaxError: invalid syntax (3802221488.py, line 180)